#### AIDP API Processing - Updated code

In [1]:
# ---------------------------------------------------------------------------
# Default OCI config read validation
# ---------------------------------------------------------------------------
import oci
COMPARTMENT_ID="<>compartment" 
config = oci.config.from_file("/Workspace/.oci/config")
oci.config.validate_config(config)
print("OCI_TENANCY: "+config["tenancy"]+"\nUSER: "+config["user"]+ \
      "\nFINGERPRINT: "+config["fingerprint"]+"\nREGION: "+config.get("region")+ \
      " KEY_FILE: "+config.get("key_file")+" KEY_CONTENT: "+ \
      str(config.get("key_content"))+" PASSPHRASE: "+str(config.get("passphrase")))
#print("OCI USER: "+config["user"])
#print("OCI FINGERPRINT: "+config["fingerprint"])
#print("OCI_REGION: "+config.get("region"))
#print("OCI_KEY_FILE: "+config.get("key_file"))
#print("OCI_KEY_CONTENT: "+str(config.get("key_content"))) #Optional (setup in private pem file)
#print("OCI_PASSPHRASE: "+str(config.get("passphrase")))   #Optional

OCI_TENANCY: <TenancyID>
USER: <UserID>
FINGERPRINT: <>Fingerprint
REGION: us-chicago-1 KEY_FILE: /Workspace/.oci/oci_api_key.pem KEY_CONTENT: None PASSPHRASE: None


In [2]:
import base64
import json
import os
import sys

# Include workspace path
from pathlib import Path
module_dir="/Workspace/Demos"
sys.path.insert(0, str(module_dir))

import config
from aidp_client import AIDPClient, AIDPAPIError, poll_until_active, print_response

# ---------------------------------------------------------------------------
# Shared client (all tests use this)
# ---------------------------------------------------------------------------

def make_client() -> AIDPClient:
    return AIDPClient(
        region=config.REGION,
        aidp_instance_id=config.AIDP_INSTANCE_ID,
        profile=config.OCI_PROFILE,
    )

In [3]:
# ---------------------------------------------------------------------------
# Test 01 — GET AIDP instance (health check / connectivity)
# ---------------------------------------------------------------------------

def test_01():
    """GET /  — verify connectivity and AIDP instance is ACTIVE."""
    print("\n=== test_01: GET AIDP instance ===")
    client = make_client()
    resp = client.get("")
    #print_response(resp, "GET instance")

    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    data = resp.json()
    state = data.get("lifecycleState", "?")
    print(f"  Instance state: {state}")
    assert state == "ACTIVE", f"Expected ACTIVE, got {state}"
    print("  PASS")
    return data
items = test_01()
print(f" InstanceId={items['id']}\tdisplayName={items['displayName']}")


=== test_01: GET AIDP instance ===


  Instance state: ACTIVE
  PASS
 InstanceId=ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaawe6j4fqasregsvzcku76sbmsuvew5stm4ozvrbf53c2usukvnwfa	displayName=Pilot


In [4]:
# ---------------------------------------------------------------------------
# Test 02 — List workspaces
# ---------------------------------------------------------------------------

def test_02():
    """GET /workspaces  — list all workspaces in the AIDP instance."""
    print("\n=== test_02: List workspaces ===")
    client = make_client()
    resp = client.get("/workspaces")
    #print_response(resp, "GET /workspaces")

    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    items = resp.json()
    if isinstance(items, dict):
        items = items.get("items", [])
    #print(f"  Found {len(items)} workspace(s):")
    #for ws in items:
        #print(f"    key={ws.get('key')}  name={ws.get('displayName')}  state={ws.get('lifecycleState')}")
    #print("  PASS")
    return items
items=test_02()
print(f"  Found {len(items)} workspace(s):")
for ws in items:
        print(f"    key={ws.get('key')}  name={ws.get('displayName')}  state={ws.get('lifecycleState')}")


=== test_02: List workspaces ===


  Found 2 workspace(s):
    key=ddc462c8-a545-4ff5-ab4f-1634af1cbe6c  name=Pvtpilot  state=ACTIVE
    key=7afddb64-736f-4308-9733-fdf755bc5bf0  name=pilot  state=ACTIVE


In [10]:
# ---------------------------------------------------------------------------
# Test 02 — 1 - Update description of a specific workspace
# ---------------------------------------------------------------------------

def test_02_1(ws_key: str, ws_name:str):
    """UPDATE Workspace based on the workspace id"""
    print("\n=== test_02_1: Update workspace description ===")
    ws = ws_key
    client = make_client()
    resp = client.put(f"/workspaces/{ws}", body={
    	"displayName": ws_name,
    	"description": f"Updated description now via REST API to {ws_name}",
		})
    
    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    print(f"status_code={resp.status_code}")
    items = resp.json()

    return items

resp=test_02_1(ws_key="7afddb64-736f-4308-9733-fdf755bc5bf0",ws_name="pilot")
print(f"key={resp['key']} display={resp['displayName']} description= {resp['description']} timeupd= {resp['timeUpdated']}")


=== test_02_1: Update workspace description ===


status_code=202
key=7afddb64-736f-4308-9733-fdf755bc5bf0 display=pilot description= Updated description now via REST API to pilot timeupd= 2026-05-21T12:03:13.782Z


In [12]:
# ---------------------------------------------------------------------------
# Test 02 — 2 - Update security permission for the user
# ---------------------------------------------------------------------------
USER="ocid1.user.oc1..aaaaaaaajz34bwifxsu6u455kya5gy2pzzaimtjy2wdmqzahcbk7prr67e3q"
client = make_client()
resp = client.post(f"/workspaces/{ws}/actions/managePermission", body={
    "grantPermissions": [
        {
            "principalId": USER,  # replace with real OCID
            "principalType": "USER",
            "permission": "MANAGER"  # VIEWER | EDITOR | MANAGER
        }
    ]
})
print(resp)

<Response [400]>


In [14]:
# ---------------------------------------------------------------------------
# Test 03 — Find existing workspace by name (PATH B)
# ---------------------------------------------------------------------------

def test_03(ws_name:str):
    """GET /workspaces then filter by displayName == EXISTING_WORKSPACE_NAME."""
    print(f"\n=== test_03: Find workspace '{config.EXISTING_WORKSPACE_NAME}' ===")
    client = make_client()
    resp = client.get("/workspaces")

    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    items = resp.json()
    if isinstance(items, dict):
        items = items.get("items", [])

    match = None
    for ws in items:
        if ws.get("displayName") == ws_name: #config.EXISTING_WORKSPACE_NAME:
            match = ws
            break

    if not match:
        #print(f"  FAIL: Workspace '{config.EXISTING_WORKSPACE_NAME}' not found.")
        print(f"  FAIL: Workspace '{ws_name}' not found.")
        print(f"  Available workspaces:")
        for ws in items:
            print(f"    {ws.get('displayName')} (key={ws.get('key')})")
        sys.exit(1)

    ws_key = match.get("key") or match.get("id")
    print(f"  Found: key={ws_key}  state={match.get('lifecycleState')}")
    #print(f"\n  >>> Copy this to config.py WORKSPACE_KEY: {ws_key}")
    #print("  PASS")
    return match

resp=test_03(ws_name="pilot")
print(f"key={resp['key']}\tdisplayname={resp['displayName']}")


=== test_03: Find workspace "pilot" ===


  Found: key=7afddb64-736f-4308-9733-fdf755bc5bf0  state=ACTIVE
key=7afddb64-736f-4308-9733-fdf755bc5bf0	displayname=pilot


In [16]:
# ---------------------------------------------------------------------------
# Test 09 - List all catalogs for the AIDP instance
# ---------------------------------------------------------------------------
def test_09():
    """GET /catalogs  — list all catalogs."""
    print(f"\n=== test_09: List catalogs ===")

    client = make_client()
    resp = client.get("/catalogs")
    #print_response(resp, "GET /catalogs")

    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    items = resp.json()
    if isinstance(items, dict):
        items = items.get("items", [])
    print(f"  Found {len(items)} catalog(s):")
    #for c in items:
        #print(f"    key={c.get('key')}  name={c.get('displayName')}  state={c.get('lifecycleState')}")
    print("  PASS")
    return items
items=test_09()
for c in items:
    print(f"    key={c.get('key')}  name={c.get('displayName')}  catalogGuid={c.get('catalogGuid')} state={c.get('lifecycleState')}")


=== test_09: List catalogs ===


  Found 6 catalog(s):
  PASS
    key=psft  name=psft  catalogGuid=179e8bfa32fc44b3893acd8e64590d78 state=ACTIVE
    key=psftcln  name=psftcln  catalogGuid=344402b3940445a4a5946b8544669218 state=ACTIVE
    key=peoplesoft  name=peoplesoft  catalogGuid=c13d3bedef084a8baab84e2fc63018d0 state=ACTIVE
    key=healthcare  name=healthcare  catalogGuid=c677e4afa63d4e758f1d2de06fa287ec state=ACTIVE
    key=vectordb26ai  name=vectordb26ai  catalogGuid=cf71a1ec83a9416a8e47b37456a013ea state=ACTIVE
    key=default  name=default  catalogGuid=hive state=ACTIVE


In [22]:
# ---------------------------------------------------------------------------
# Test 10 - List of schemas for a particular catalog
# ---------------------------------------------------------------------------
def test_10(ws_key:str):
    """GET /jobs  — list all jobs for an workspace."""
    print(f"\n=== test_09: List catalogs ===")
    ws=ws_key
    client = make_client()
    resp = client.get(f"/workspaces/{ws}/jobs")

    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    items = resp.json()
    if isinstance(items, dict):
        items = items.get("items", [])
    print(f"  Found {len(items)} catalog(s):")
    #for c in items:
        #print(f"    key={c.get('key')}  name={c.get('displayName')}  state={c.get('lifecycleState')}")
    print("  PASS")
    return items    

resp=test_10(ws_key="7afddb64-736f-4308-9733-fdf755bc5bf0")
for job in resp:
    print(
        f"key={job['key']} "
        f"display={job['name']} "
        f"description={job.get('description', '')} "
        f"timeupd={job['timeUpdated']}"
    )


=== test_09: List catalogs ===


  Found 5 catalog(s):
  PASS
key=f511f5e2-d255-4805-81dc-df58a4a58cc2 display=test.job description= timeupd=2026-04-29T16:25:20.663Z
key=cb50b351-77a8-42ce-9e90-d1138c7984b1 display=BronzeOS.job description= timeupd=2026-05-08T15:52:06.135Z
key=fce6b7be-91ae-4fc3-a1e4-e6e030a4fc4a display=BronzeTable.job description= timeupd=2026-05-08T16:01:38.524Z
key=332d22e6-5041-49da-b3df-4ea88a4964c2 display=BronzeVolume.job description= timeupd=2026-05-08T17:04:10.417Z
key=093759ab-4c31-4bac-a390-2a3301ae9e55 display=ParamTest.job description= timeupd=2026-05-08T18:08:04.668Z


In [36]:
# ---------------------------------------------------------------------------
# Test 11 - List job details
# ---------------------------------------------------------------------------
def test_11(ws_key:str, job_key:str):
    """GET /jobs  — list all jobs for an workspace."""
    
    print(f"\n=== test_11: List job details ===")
    ws=ws_key
    jk=job_key
    client = make_client()
    resp = client.get(f"/workspaces/{ws}/jobs/{jk}")

    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    items = resp.json()

    print("  PASS")
    return items     

ws_key="7afddb64-736f-4308-9733-fdf755bc5bf0"
job_key="093759ab-4c31-4bac-a390-2a3301ae9e55"
job = test_11(ws_key=ws_key,job_key=job_key)
print(
    	f"key={resp.get('key')} "
    	f"name={job.get('name')} "
    	f"description={job.get('description')} "
    	f"path={job.get('path')} "
    	f"timeCreated={job.get('timeCreated')} "
    	f"timeUpdated={job.get('timeUpdated')}"
)


=== test_11: List job details ===


  PASS
key=093759ab-4c31-4bac-a390-2a3301ae9e55 name=ParamTest.job description= path=/Workspace/jobs timeCreated=2026-05-08T18:08:04.668Z timeUpdated=2026-05-15T17:55:56.326Z
